# Model Fine-Tuning with Unsloth

This notebook demonstrates fine-tuning Qwen 2.5 14B for native ads detection using Unsloth.

## Prerequisites

```bash
pip install -r ../langchain-refactor/requirements-finetuning.txt
```

## Setup

In [ ]:
import sys
sys.path.append('../langchain-refactor')

from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json
from pathlib import Path

## Configuration

In [ ]:
# Model configuration
BASE_MODEL = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
DATASET_PATH = "../data/llm_dataset_mixed_json.json"
OUTPUT_DIR = "../models/native-ads-qwen14b-v2"

# Training configuration
MAX_STEPS = 2000
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
EVAL_STEPS = 100

print(f"Base Model: {BASE_MODEL}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Max Steps: {MAX_STEPS}")

## Load Model with Unsloth

In [ ]:
print("Loading model with Unsloth optimization...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("✅ Model loaded!")

## Add LoRA Adapters

In [ ]:
print("Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ LoRA adapters added!")

## Load and Prepare Dataset

In [ ]:
# Load dataset
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")

# Format for training
from prompts.classification_prompts import SIMPLE_LOCAL_PROMPT_TEMPLATE

formatted_data = []
for sample in data:
    try:
        output_data = json.loads(sample['output'])
        expected_output = json.dumps(output_data, ensure_ascii=False)
    except:
        expected_output = sample['output']
    
    text = f"""Klasifikasikan berita berikut sebagai "native ads" atau "berita murni".

Native Ads adalah konten yang MENGGABUNGKAN semua ciri berikut:
1. Nada positif/netral (tidak mengkritik subjek)
2. Bahasa persuasif (mengajak/meyakinkan)
3. Mempromosikan produk/brand/instansi
4. Hanya satu sudut pandang (tidak objektif)

Konten:
{sample['input']}

Output (JSON):
{expected_output}"""
    
    formatted_data.append({"text": text})

# Create dataset
dataset = Dataset.from_list(formatted_data)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training samples: {len(split['train'])}")
print(f"Evaluation samples: {len(split['test'])}")

## Setup Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_steps=EVAL_STEPS * 2,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
    ),
)

print("✅ Trainer configured!")

## Start Training

⚠️ This will take several hours depending on your GPU

In [ ]:
print("Starting training...")
print("="*80)

trainer_stats = trainer.train()

print("\n" + "="*80)
print("Training complete!")
print(f"Total time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final train loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"Final eval loss: {trainer_stats.metrics.get('eval_loss', 'N/A')}")

## Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save merged model (16-bit)
merged_path = f"{OUTPUT_DIR}_merged_16bit"
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit"
)

print(f"✅ Model saved to:")
print(f"   LoRA adapters: {OUTPUT_DIR}")
print(f"   Merged 16-bit: {merged_path}")

## Next Steps

Model is ready for evaluation! See `03_model_evaluation.ipynb`